In [ ]:
import tifffile as tf 
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np 
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
!pip install imblearn
from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_validate
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
import pickle 
import statistics 


In [ ]:
key = "" # sample identifier

In [ ]:
pixel_clustering_path = r"" # path to pixel clustering results

In [ ]:
with open(pixel_clustering_path + '/cluster_color_dict.pickle', 'rb') as f: 
    cluster_color_dict = pickle.load(f)

In [ ]:
glomerulus_classification_path = r"" # path to glomerulosclerosis results

In [ ]:
with open(glomerulus_classification_path + '/sample_label.pickle', 'rb') as f: 
    sample_labels = pickle.load(f)

In [ ]:
data_frequency_df_dropped = pd.read_csv(glomerulus_classification_path + '/data_frequency_df.csv')
data_frequency_df_dropped = data_frequency_df_dropped.drop(index=170)
X_freq = np.array(data_frequency_df_dropped.iloc[:, 1:-1])
y = np.array(data_frequency_df_dropped.iloc[:, -1])

In [ ]:
# non sample level split
"""lf = LogisticRegression() 
rf = RandomForestClassifier()
mlp = MLPClassifier()

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oversample_pipeline_rf = Pipeline([('oversample', ADASYN()), ('model', rf)])
oversample_pipeline_lf = Pipeline([('oversample', ADASYN()), ('model', lf)])
oversample_pipeline_mlp = Pipeline([('oversample', ADASYN()), ('model', mlp)])

scoring = ['roc_auc', 'f1', 'precision', 'recall', 'accuracy'] 

cv_results_oversample_rf = cross_validate(oversample_pipeline_rf, X_freq, y, cv=skf, scoring=scoring, return_estimator=True)
cv_results_oversample_lf = cross_validate(oversample_pipeline_lf, X_freq, y, cv=skf, scoring=scoring, return_estimator=True)
cv_results_oversample_mlp = cross_validate(oversample_pipeline_mlp, X_freq, y, cv=skf, scoring=scoring, return_estimator=True)"""

In [ ]:
# sample level split
lf = LogisticRegression()
rf = RandomForestClassifier()
mlp = MLPClassifier()

sgkf = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

oversample_pipeline_rf = Pipeline([
    ('oversample', ADASYN()),
    ('model', rf)
])

oversample_pipeline_lf = Pipeline([
    ('oversample', ADASYN()),
    ('model', lf)
])

oversample_pipeline_mlp = Pipeline([
    ('oversample', ADASYN()),
    ('model', mlp)
])

scoring = ['roc_auc', 'f1', 'precision', 'recall', 'accuracy']

cv_results_oversample_rf = cross_validate(
    oversample_pipeline_rf,
    X_freq,
    y,
    cv=sgkf,
    groups=sample_labels,
    scoring=scoring,
    return_estimator=True
)

cv_results_oversample_lf = cross_validate(
    oversample_pipeline_lf,
    X_freq,
    y,
    cv=sgkf,
    groups=sample_labels,
    scoring=scoring,
    return_estimator=True
)

cv_results_oversample_mlp = cross_validate(
    oversample_pipeline_mlp,
    X_freq,
    y,
    cv=sgkf,
    groups=sample_labels,
    scoring=scoring,
    return_estimator=True
)

In [ ]:
fold_assignments = np.full(len(y), -1)

for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(X_freq, y, groups=sample_labels)
):
    fold_assignments[test_idx] = fold


In [ ]:
fold_assignments

In [ ]:
with open(glomerulus_classification_path + '/rf_model_auc_by_sample.pickle', 'wb') as f: 
    pickle.dump(cv_results_oversample_rf['test_roc_auc'], f)

with open(glomerulus_classification_path + '/lf_model_auc_by_sample.pickle', 'wb') as f: 
    pickle.dump(cv_results_oversample_lf['test_roc_auc'], f)

with open(glomerulus_classification_path + '/mlp_model_auc_by_sample.pickle', 'wb') as f: 
    pickle.dump(cv_results_oversample_mlp['test_roc_auc'], f)

In [ ]:
with open(glomerulus_classification_path + '/rf_model_f1_by_sample.pickle', 'wb') as f: 
    pickle.dump(cv_results_oversample_rf['test_f1'], f)

with open(glomerulus_classification_path + '/lf_model_f1_by_sample.pickle', 'wb') as f: 
    pickle.dump(cv_results_oversample_lf['test_f1'], f)

with open(glomerulus_classification_path + '/mlp_model_f1_by_sample.pickle', 'wb') as f: 
    pickle.dump(cv_results_oversample_mlp['test_f1'], f)

In [ ]:
with open(glomerulus_classification_path + '/rf_model_acc_by_sample.pickle', 'wb') as f: 
    pickle.dump(cv_results_oversample_rf['test_accuracy'], f)

with open(glomerulus_classification_path + '/lf_model_acc_by_sample.pickle', 'wb') as f: 
    pickle.dump(cv_results_oversample_lf['test_accuracy'], f)

with open(glomerulus_classification_path + '/mlp_model_acc_by_sample.pickle', 'wb') as f: 
    pickle.dump(cv_results_oversample_mlp['test_accuracy'], f)

In [ ]:
auc_mean_rf = statistics.mean(cv_results_oversample_rf['test_roc_auc'])
f1_mean_rf = statistics.mean(cv_results_oversample_rf['test_f1'])
accuracy_mean_rf = statistics.mean(cv_results_oversample_rf['test_accuracy'])

auc_mean_lf = statistics.mean(cv_results_oversample_lf['test_roc_auc'])
f1_mean_lf = statistics.mean(cv_results_oversample_lf['test_f1'])
accuracy_mean_lf = statistics.mean(cv_results_oversample_lf['test_accuracy'])

auc_mean_mlp = statistics.mean(cv_results_oversample_mlp['test_roc_auc'])
f1_mean_mlp = statistics.mean(cv_results_oversample_mlp['test_f1'])
accuracy_mean_mlp = statistics.mean(cv_results_oversample_mlp['test_accuracy'])

In [ ]:
auc_err_rf = statistics.stdev(cv_results_oversample_rf['test_roc_auc'])/np.sqrt(cv_results_oversample_rf['test_roc_auc'].shape[0])
f1_err_rf = statistics.stdev(cv_results_oversample_rf['test_f1'])/np.sqrt(cv_results_oversample_rf['test_f1'].shape[0])
accuracy_err_rf = statistics.stdev(cv_results_oversample_rf['test_accuracy'])/np.sqrt(cv_results_oversample_rf['test_accuracy'].shape[0])

auc_err_lf = statistics.stdev(cv_results_oversample_lf['test_roc_auc'])/np.sqrt(cv_results_oversample_lf['test_roc_auc'].shape[0])
f1_err_lf = statistics.stdev(cv_results_oversample_lf['test_f1'])/np.sqrt(cv_results_oversample_lf['test_f1'].shape[0])
accuracy_err_lf = statistics.stdev(cv_results_oversample_lf['test_accuracy'])/np.sqrt(cv_results_oversample_lf['test_accuracy'].shape[0])

auc_err_mlp = statistics.stdev(cv_results_oversample_mlp['test_roc_auc'])/np.sqrt(cv_results_oversample_mlp['test_roc_auc'].shape[0])
f1_err_mlp = statistics.stdev(cv_results_oversample_mlp['test_f1'])/np.sqrt(cv_results_oversample_mlp['test_f1'].shape[0])
accuracy_err_mlp = statistics.stdev(cv_results_oversample_mlp['test_accuracy'])/np.sqrt(cv_results_oversample_mlp['test_accuracy'].shape[0])

In [ ]:
metrics = ['Accuracy', 'AUC-ROC', 'F1']

In [ ]:
model_means = {'Random Forest': (accuracy_mean_rf, auc_mean_rf, f1_mean_rf), 
          'Logistic Regression': (accuracy_mean_lf, auc_mean_lf, f1_mean_lf), 
          'Multi-Layer Perceptron': (accuracy_mean_mlp, auc_mean_mlp, f1_mean_mlp)}

In [ ]:
model_means

In [ ]:
model_errs = {'Random Forest': (accuracy_err_rf, auc_err_rf, f1_err_rf), 
          'Logistic Regression': (accuracy_err_lf, auc_err_lf, f1_err_lf), 
          'Multi-Layer Perceptron': (accuracy_err_mlp, auc_err_mlp, f1_err_mlp)}

In [ ]:
with open(glomerulus_classification_path + '/model_means.pickle', 'wb') as f: 
    pickle.dump(model_means, f)

In [ ]:
with open(glomerulus_classification_path + '/model_errs.pickle', 'wb') as f: 
    pickle.dump(model_errs, f)

In [ ]:
#Accuracy
print(stats.ttest_ind(cv_results_oversample_mlp['test_accuracy'], cv_results_oversample_lf['test_accuracy'], equal_var=True)[1])
print(stats.ttest_ind(cv_results_oversample_mlp['test_accuracy'], cv_results_oversample_rf['test_accuracy'], equal_var=True)[1])
print(stats.ttest_ind(cv_results_oversample_lf['test_accuracy'], cv_results_oversample_rf['test_accuracy'], equal_var=True)[1])

#AUC-ROC
print(stats.ttest_ind(cv_results_oversample_mlp['test_roc_auc'], cv_results_oversample_lf['test_roc_auc'], equal_var=True)[1])
print(stats.ttest_ind(cv_results_oversample_mlp['test_roc_auc'], cv_results_oversample_rf['test_roc_auc'], equal_var=True)[1])
print(stats.ttest_ind(cv_results_oversample_lf['test_roc_auc'], cv_results_oversample_rf['test_roc_auc'], equal_var=True)[1])

#F1
print(stats.ttest_ind(cv_results_oversample_mlp['test_f1'], cv_results_oversample_lf['test_f1'], equal_var=True)[1])
print(stats.ttest_ind(cv_results_oversample_mlp['test_f1'], cv_results_oversample_rf['test_f1'], equal_var=True)[1])
print(stats.ttest_ind(cv_results_oversample_lf['test_f1'], cv_results_oversample_rf['test_f1'], equal_var=True)[1])


In [ ]:
colors = {'Random Forest': 'r', 'Logistic Regression': 'b', 'Multi-Layer Perceptron': 'orange'}

In [ ]:
import matplotlib as mpl 
mpl.rcParams["font.size"] = 15

fig, ax = plt.subplots(1)
fig.set_figheight(6)
fig.set_figwidth(6)

x = np.arange(len(metrics))  
width = 0.25  
multiplier = 0

i = 0
for attribute, measurement in model_means.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, measurement, width, label=attribute, color=colors[attribute])
    ax.errorbar(x+offset, measurement, model_errs[attribute], fmt='none', ecolor='black', capsize=5)
    multiplier += 1


ax.set_xlabel('Metrics')

ax.set_ylabel('Performance')
ax.set_xticks(x + width, metrics)

In [ ]:
import matplotlib.patches as mpatches
fig, ax = plt.subplots(1)

i = 0
patches = [
    mpatches.Patch(color=color, label=label)
    for label, color in zip(list(colors.keys()), list(colors.values()))]
ax.legend(patches, list(colors.keys()), loc='center', frameon=False)
ax.axis('off')
i+=1
plt.show()

In [ ]:
#feature importance for non sample level split (change split to sample level for feature importance for sample level split)
"""feature_importance = []
for (train_idx, test_idx), est in zip(skf.split(X_freq, y), cv_results_oversample_mlp["estimator"]):
    X_test, y_test = X_freq[test_idx], y[test_idx]

    imp = permutation_importance(
        est, X_test, y_test,
        n_repeats=10,
        scoring='f1'
    )
    feature_importance.append(imp.importances_mean)"""

In [ ]:
"""from sklearn.inspection import permutation_importance

feature_importance = []

for (train_idx, test_idx), est in zip(
    sgkf.split(X_freq, y, groups=sample_labels),
    cv_results_oversample_mlp["estimator"]
):
    X_test = X_freq[test_idx]
    y_test = y[test_idx]

    imp = permutation_importance(
        est,
        X_test,
        y_test,
        n_repeats=10,
        scoring='f1',
        random_state=42
    )

    feature_importance.append(imp.importances_mean)"""

In [ ]:
"""feature_importance = np.array(feature_importance)"""

In [ ]:
"""with open(glomerulus_classification_path + '/feature_importances.pickle', 'wb') as f: 
    pickle.dump(feature_importance, f)"""

In [ ]:
"""import matplotlib.pyplot as plt
import numpy as np

mean_importance = np.mean(feature_importance, axis=0)
sorted_idx = np.argsort(mean_importance)[::-1]

top = 10
idx = sorted_idx[:top]

fig, ax = plt.subplots(figsize=(5,4))

ax.bar(
    np.arange(top),
    mean_importance[idx],
    color=np.array(list(cluster_color_dict.values()))[idx],
    alpha=0.7
)

# overlay individual fold values
for fold_imp in feature_importance:
    ax.scatter(
        np.arange(top),
        fold_imp[idx],
        color='black',
        s=20,
        zorder=3
    )

ax.set_xticks(np.arange(top))
ax.set_xticklabels(idx + 1)
ax.set_xlabel("Cluster")
ax.set_ylabel("Permutation Importance")
plt.tight_layout()
plt.show()"""